In [1]:
# --- iPython Config --- #
from IPython import get_ipython
if 'IPython.extensions.autoreload' not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic('load_ext', 'autoreload')
else:
    get_ipython().run_line_magic('reload_ext', 'autoreload')
%autoreload 2

# --- System and Path --- #
import os
import sys
REPO_PATH = os.path.abspath(os.path.join('..'))
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)
import warnings
warnings.filterwarnings("ignore")

# --- Data Manipulation --- #
import pandas as pd
import numpy as np

In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Advanced Short-Answer Scoring with TF–IDF + Thai BERT embeddings + AutoGluon Ensemble
Author: Your Name

Description:
  This code demonstrates an advanced approach to short-answer scoring by:
    1. Reading train/test CSVs
    2. Combining question + answer text
    3. Generating TF–IDF vectors and Thai BERT embeddings
    4. Concatenating them for a unified feature representation
    5. Training an ensemble of models using AutoGluon
    6. Evaluating via cross-validation
    7. Outputting predictions to submission.csv

Dependencies (install with pip):
    scikit-learn, transformers, pythainlp, tqdm, autogluon
Usage:
    python advanced_shortanswer.py
"""

import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModel  # For Thai BERT
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from autogluon.tabular import TabularPredictor  # AutoGluon Ensemble

###############################################################################
# 1) Reading Data
###############################################################################
def read_data():
    train_df = pd.read_csv(os.path.join(REPO_PATH, "data", "train.csv"))
    test_df = pd.read_csv(os.path.join(REPO_PATH, "data", "test.csv"))
    sub_df = pd.read_csv(os.path.join(REPO_PATH, "data", "sample_submission.csv"))
    return train_df, test_df, sub_df

###############################################################################
# 2) Text Combination
###############################################################################
def combine_text(question, answer):
    q = question if isinstance(question, str) else ""
    a = answer if isinstance(answer, str) else ""
    return q + " " + a

###############################################################################
# 3) TF–IDF Vectorizer
###############################################################################
def build_tfidf_vectorizer():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=10000)

###############################################################################
# 4) Thai BERT Embeddings
###############################################################################
class ThaiBERTEmbedder:
    def __init__(self, model_name="airesearch/wangchanberta-base-att-spm-uncased", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Loading BERT tokenizer/model for: {model_name} on device: {self.device}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def encode(self, text_list, batch_size=16, max_length=128):
        all_embs = []
        for i in range(0, len(text_list), batch_size):
            batch_text = text_list[i : i + batch_size]
            inputs = self.tokenizer(
                batch_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
            cls_emb = outputs.last_hidden_state[:, 0, :]
            all_embs.append(cls_emb.cpu().numpy())
        return np.concatenate(all_embs, axis=0)

###############################################################################
# 5) Preprocess: TF–IDF + BERT
###############################################################################
def preprocess_data(train_df, test_df, tfidf_vectorizer, bert_embedder):
    train_df["text"] = train_df.apply(lambda r: combine_text(r["question"], r["answer"]), axis=1)
    test_df["text"] = test_df.apply(lambda r: combine_text(r["question"], r["answer"]), axis=1)

    print("Fitting TF–IDF on training data...")
    X_tfidf_train = tfidf_vectorizer.fit_transform(train_df["text"].tolist()).toarray()
    print("Transforming test data with TF–IDF...")
    X_tfidf_test = tfidf_vectorizer.transform(test_df["text"].tolist()).toarray()

    print("Generating BERT embeddings for train...")
    X_bert_train = bert_embedder.encode(train_df["text"].tolist())
    print("Generating BERT embeddings for test...")
    X_bert_test = bert_embedder.encode(test_df["text"].tolist())

    X_train = np.hstack([X_tfidf_train, X_bert_train])
    X_test = np.hstack([X_tfidf_test, X_bert_test])
    y_train = train_df["score"].values

    return X_train, y_train, X_test

###############################################################################
# 6) Train AutoGluon Ensemble
###############################################################################
def train_ensemble(X_train, y_train):
    train_data = pd.DataFrame(X_train)
    train_data["score"] = y_train
    predictor = TabularPredictor(label="score", problem_type="regression").fit(train_data)
    return predictor

###############################################################################
# 7) Predict & Save Submission
###############################################################################
def predict_and_save(predictor, X_test, sub_df, output_name="submission.csv"):
    test_data = pd.DataFrame(X_test)
    predictions = predictor.predict(test_data)
    sub_df["score"] = predictions
    sub_df.to_csv(output_name, index=False)
    print(f"Submission saved to: {output_name}")

###############################################################################
# Main Script
###############################################################################
def main():
    print("=== (1) Reading data ===")
    train_df, test_df, sub_df = read_data()

    print("=== (2) Building TF–IDF vectorizer ===")
    tfidf_vectorizer = build_tfidf_vectorizer()

    print("=== (3) Initializing Thai BERT embedder ===")
    bert_embedder = ThaiBERTEmbedder()

    print("=== (4) Preprocessing: TF–IDF + BERT embeddings ===")
    X_train, y_train, X_test = preprocess_data(train_df, test_df, tfidf_vectorizer, bert_embedder)

    print("=== (5) Train AutoGluon Ensemble ===")
    predictor = train_ensemble(X_train, y_train)

    print("=== (6) Predict & Save Submission ===")
    predict_and_save(predictor, X_test, sub_df)

    print("All done! Check submission.csv for your predictions.")

if __name__ == "__main__":
    main()


=== (1) Reading data ===
=== (2) Building TF–IDF vectorizer ===
=== (3) Initializing Thai BERT embedder ===
Loading BERT tokenizer/model for: airesearch/wangchanberta-base-att-spm-uncased on device: cpu
=== (4) Preprocessing: TF–IDF + BERT embeddings ===
Fitting TF–IDF on training data...
Transforming test data with TF–IDF...
Generating BERT embeddings for train...
Generating BERT embeddings for test...


No path specified. Models will be saved in: "AutogluonModels/ag-20250307_112044"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.11.10
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.3.0: Thu Jan  2 20:23:36 PST 2025; root:xnu-11215.81.4~3/RELEASE_ARM64_T8112
CPU Count:          8
Memory Avail:       1.42 GB / 8.00 GB (17.7%)
Disk Space Avail:   24.13 GB / 228.27 GB (10.6%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. 

=== (5) Train AutoGluon Ensemble ===


Beginning AutoGluon training ...
AutoGluon will save models to "/Users/pupipatsingkhorn/Developer/repositories/NLP/nlp-2025-midterm-kaggle-asas/notebooks/AutogluonModels/ag-20250307_112044"
Train Data Rows:    362
Train Data Columns: 3961
Label Column:       score
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    1462.00 MB
	Train Data (Original)  Memory Usage: 10.94 MB (0.7% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Unused Original Features (C

=== (6) Predict & Save Submission ===
Submission saved to: submission.csv
All done! Check submission.csv for your predictions.
